In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import pylab
from PIL import Image
from math import e
import astropy.io.fits as pf
from astropy.io import fits
from mpl_toolkits.axes_grid1 import make_axes_locatable
from tqdm import tqdm
from astropy.convolution import convolve
from astropy.convolution import Gaussian2DKernel
from scipy.optimize import curve_fit
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from astropy.coordinates import ICRS    
from astropy.coordinates import Galactic 
from astropy.coordinates import SkyCoord 

In [ ]:
def fix_drao_header(hdu):
    hdu[0].header['NAXIS'] = 2
    try:
        del(hdu[0].header['NAXIS3'])
        del(hdu[0].header['NAXIS4'])
    except:
        pass
    del(hdu[0].header['CTYPE3'])
    del(hdu[0].header['CTYPE4'])
    del(hdu[0].header['CRVAL3'])
    del(hdu[0].header['CRVAL4'])
    del(hdu[0].header['CRPIX3'])
    del(hdu[0].header['CRPIX4'])
    del(hdu[0].header['CDELT3'])
    del(hdu[0].header['CDELT4'])
    del(hdu[0].header['CROTA3'])
    del(hdu[0].header['CROTA4'])
    try:
        del(hdu[0].header['FILENAME'])
        del(hdu[0].header['BSCALE'])
        del(hdu[0].header['BLOCKED '])
        del(hdu[0].header['BZERO'])
        del(hdu[0].header['DATE'])
    except:
        pass
    try:
        del(hdu[0].header['DATAMIN'])          
        del(hdu[0].header['DATAMAX'])
        del(hdu[0].header['MINCOL'])          
        del(hdu[0].header['MAXCOL'])       
        del(hdu[0].header['MINROW'])             
        del(hdu[0].header['MAXROW'])
        del(hdu[0].header['EQUINOX'])
        del(hdu[0].header['BANDW'])
        del(hdu[0].header['OBSRA'])        
        del(hdu[0].header['OBSDEC'])
        del(hdu[0].header['UVGRID'])             
        del(hdu[0].header['BLGRAD'])
        del(hdu[0].header['POLCODE'])                                                                                        
        del(hdu[0].header['MAXBAS'])      
        del(hdu[0].header['MINBAS'])
    except:
        pass                                                                                                                                                                      
    del(hdu[0].header['OBJECT'])                           
    del(hdu[0].header['ORIGIN'])           
    del(hdu[0].header['INSTRUME'])           
    del(hdu[0].header['OBSERVER'])                    
    del(hdu[0].header['DATE-OBS'])
    del(hdu[0].header['CROTA2'])          
    del(hdu[0].header['OBSFREQ'])                                                                                        
    del(hdu[0].header['HISTORY'])      

    hdu[0].header['CUNIT1'] = 'deg'
    hdu[0].header['CUNIT2'] = 'deg'
    hdu[0].header['BUNIT'] = 'K'
    try:
        hdu[0].data = hdu[0].data[0,0,:,:]
    except:
        hdu[0].data = hdu[0].data
    
    return hdu

### Get the full-scale CGPS (W06+WSRT+CGPS) data

In [ ]:
hduU1 = fits.open('/srv/data/cgps/continuum/CGPS_MEY2_1420_MHz_U_image.fits')
mapU1 = hduU1[0].data[0,0]

hduU2 = fits.open('/srv/data/cgps/continuum/CGPS_MEY1_1420_MHz_U_image.fits')
mapU2 = hduU2[0].data[0,0]

hduU1_fix = fix_drao_header(hduU1)
hduU2_fix = fix_drao_header(hduU2)

hdr_new = hduU1_fix[0].header.copy()
hdr_new['NAXIS2'] = 2048
hdr_new['NAXIS1'] = 1024
hdr_new['CRPIX2'] = 1025
hdr_new['CRPIX1'] = 513 
hdr_new['CRVAL2'] = 1.
hdr_new['CRVAL1'] = 160.75

mapU_cgps, footprint = reproject_and_coadd([hduU1_fix,hduU2_fix],hdr_new,reproject_function=reproject_interp)

wcs_cgps = WCS(hdr_new)
print(wcs_cgps)
hdr_cgps = hdr_new.copy()

dxy = abs(hdr_new['CDELT2'])
print(dxy)

### Get the "full-scale" ST+HBN data (NOT used in the figure - just a quick comparison here)

In [ ]:
mapsU_ST_HBN = []

for band in ['a','b','c','d']:

    print(band)
    
    hdu = fits.open('/srv/data/cgps-gmims/cgps_for_sims/mey1'+band+'u_CGPS_GMIMS_image.fits')
    hdu_mey1 = fix_drao_header(hdu)
    hdr_new = hdu_mey1[0].header.copy()
    hdr_new['NAXIS2'] = 2048
    hdr_new['NAXIS1'] = 1024
    hdr_new['CRPIX2'] = 1025
    hdr_new['CRPIX1'] = 513 
    hdr_new['CRVAL2'] = 1.
    hdr_new['CRVAL1'] = 160.75

    hdu = fits.open('/srv/data/cgps-gmims/cgps_for_sims/mey2'+band+'u_CGPS_GMIMS_image.fits')
    hdu_mey2 = fix_drao_header(hdu)

    map_full, footprint = reproject_and_coadd([hdu_mey1,hdu_mey2],hdr_new,reproject_function=reproject_interp)

    mapsU_ST_HBN.append(map_full)

### Get the "full-scale" ST+W06 data

In [ ]:
hduU1 = fits.open('/home/aordog/Dropbox/DRAO_export/W06_ST/mey1abcdu_CGPS_W06_image.fits')
mapU1 = hduU1[0].data[0,0]

hduU2 = fits.open('/home/aordog/Dropbox/DRAO_export/W06_ST/mey2abcdu_CGPS_W06_image.fits')
mapU2 = hduU2[0].data[0,0]

hduU1_fix = fix_drao_header(hduU1)
hduU2_fix = fix_drao_header(hduU2)

hdr_new = hduU1_fix[0].header.copy()
hdr_new['NAXIS2'] = 2048
hdr_new['NAXIS1'] = 1024
hdr_new['CRPIX2'] = 1025
hdr_new['CRPIX1'] = 513 
hdr_new['CRVAL2'] = 1.
hdr_new['CRVAL1'] = 160.75

mapU_w06_st, footprint = reproject_and_coadd([hduU1_fix,hduU2_fix],hdr_new,reproject_function=reproject_interp)

wcs_w06_st = WCS(hdr_new)
print(wcs_w06_st)

#plt.imshow(mapU_w06_st,origin='lower',vmin=-1,vmax=1)
#print(mapU_w06_st[500:500+1024,:].shape)

### Check the maps (full, W06+ST, 4 channels of HBN+ST)

In [ ]:
fig, ax = plt.subplots(1,6,figsize=(12,4))

ax[0].imshow(mapU_cgps, origin='lower', vmin=-0.5, vmax=0.5, cmap='rainbow')
ax[1].imshow(mapU_w06_st, origin='lower', vmin=-0.5, vmax=0.5, cmap='rainbow')
ax[2].imshow(mapsU_ST_HBN[0], origin='lower', vmin=-0.5, vmax=0.5, cmap='rainbow')
ax[3].imshow(mapsU_ST_HBN[1], origin='lower', vmin=-0.5, vmax=0.5, cmap='rainbow')
ax[4].imshow(mapsU_ST_HBN[2], origin='lower', vmin=-0.5, vmax=0.5, cmap='rainbow')
ax[5].imshow(mapsU_ST_HBN[3], origin='lower', vmin=-0.5, vmax=0.5, cmap='rainbow')

### Check the diffs between full and W06+ST

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,6))

ax[0].imshow(mapU_cgps, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')
ax[1].imshow(mapU_w06_st, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')
ax[2].imshow(mapU_cgps-mapU_w06_st, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')

### Get the W06 data

In [ ]:
hdu = fits.open('/srv/data/cgps/drao26m_2006/u_gal.fit')
mapU_w06 = hdu[0].data/1e3
hdr_new  = hdu[0].header.copy()
wcs_w06  = WCS(hdr_new)
print(wcs_w06)

mapU_w06_rgrd, footprint = reproject_interp((mapU_w06, hdr_new), hdr_cgps)
plt.imshow(mapU_w06_rgrd)

### Get the ST-only data

In [ ]:
mapsU_st = []
for band in ['A','B','C','D']:
    print(band)
    hdu = fits.open('/srv/data/cgps-gmims/conv_regrid/Q'+band+'_C_conv4_regrd.fits')
    mapsU_st.append(hdu[0].data)

wcs_st = WCS(hdu[0].header)
print(wcs_st)

mapU_st = (mapsU_st[0]+mapsU_st[1]+mapsU_st[2]+mapsU_st[3])/4


In [ ]:
def gf(mu,fwhm,x):
    return np.exp(-4*np.log(2)*((x-mu)**2)/fwhm**2)

In [ ]:
def convolve_to_beam(map, biggest_beam, current_beam,dxy, output = False):
    
    fwhm = np.sqrt(biggest_beam**2 - current_beam**2) * 60

    if output: print('Convolving', np.round(current_beam,2), 'deg to', np.round(biggest_beam,2), 'deg\t FWHM of kernel: '+str(fwhm/60)+' degrees')   
    pixsize = dxy*60
    
    if fwhm < pixsize:
        if output: print("Just returning the map without change")
        return map
    
    if output: print('pixel size: '+str(pixsize)+' arcmin')   
    stddev = np.round((fwhm/2.355)/pixsize,0)
    if output: print('standard deviation of kernel: '+str(stddev)+' pixels')
    
    if stddev == 0:
        return map

    kernel = Gaussian2DKernel(x_stddev=stddev,y_stddev=stddev)
    
    return convolve(map,kernel), kernel
    

In [ ]:
def get_uv_axis(image, dxy, freqMHz=1420, method='right'):

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy

    if method == 'right':
        #uv_m = 2*uv_freq_m*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        #uv_n = 2*uv_freq_n*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        uv_m = uv_freq_m*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
        uv_n = uv_freq_n*(3e8/(freqMHz*1e6))*180/np.pi # corrected Oct 2024
    if method == 'wrong':
        uv_m = uv_freq_m*(3e8/(freqMHz*1e6))*180/(np.pi**2)
        uv_n = uv_freq_n*(3e8/(freqMHz*1e6))*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    uv = {}
    uv['uvm'] = uv_m
    uv['uvn'] = uv_n
    uv['xuv'] = xuv
    uv['yuv'] = yuv
    uv['ruv'] = ruv
    
    return uv
    

In [ ]:
def get_uvbounds_idx(uv,maxu,maxv):

    du = uv['uvm'][1] - uv['uvm'][0]
    dv = uv['uvn'][1] - uv['uvn'][0]

    print(du,dv)

    idx_u1 = abs(uv['uvm'] + maxu).argmin()
    idx_u2 = abs(uv['uvm'] - maxu).argmin()
    idx_v1 = abs(uv['uvn'] + maxv).argmin()
    idx_v2 = abs(uv['uvn'] - maxv).argmin()

    print(uv['uvm'][idx_u1],uv['uvm'][idx_u2])
    print(uv['uvn'][idx_v1],uv['uvn'][idx_v2])

    idx = {}
    idx['u1'] = idx_u1
    idx['u2'] = idx_u2
    idx['v1'] = idx_v1
    idx['v2'] = idx_v2

    extent = [uv['uvm'][idx_u1]-du/2, uv['uvm'][idx_u2]+du/2,
              uv['uvn'][idx_v1]-dv/2, uv['uvn'][idx_v2]+dv/2]
    
    return idx, extent

In [ ]:
def make_uv_mask(filter, R1, R2, ruv):

    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.

    if filter == 'hp':
        
        print('Applying high pass filter')
        mask = 0.5*np.sin(b*(ruv - r0))+0.5
        mask[ruv >= R2] = 1
        mask[ruv <= R1] = 0

    if filter == 'lp':
        
        print('Applying low pass filter')
        mask = -0.5*np.sin(b*(ruv - r0))+0.5
        mask[ruv >= R2] = 0
        mask[ruv <= R1] = 1

    return mask

In [ ]:
def ST_observe(image, dxy, pix0, R1=0, R2=13, bounds=300):

    # Fourier transform full-scale image:
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    # Fourier transform actual ST image:  
    #image_FFT_true = np.fft.fft2(image_ST_true)
    #image_FFT_true_shift = np.fft.fftshift(image_FFT_true)

    # Generate uv-plane axes and high-pass mask
    uv = get_uv_axis(image, dxy)
    idx, extent = get_uvbounds_idx(uv,bounds,bounds)
    mask = make_uv_mask('hp', R1, R2, uv['ruv'])

    # Apply the high-pass mask and Fourier transform back to image:
    image_FFT_shift_mask = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift_mask))
    
    return image_ST, mask

In [ ]:
def SA_observe(image, dxy, pix0, R=9, res1=1, res2=36, plots=False, bounds=300,method='fourier'):

    # Fourier transform full-scale image:
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    # Generate uv-plane axes and Gaussian beam low-pass mask:
    uv = get_uv_axis(image, dxy)
    idx, extent = get_uvbounds_idx(uv,bounds,bounds) 
    #print(uv['ruv'])
    mask = gf(0,2*R,uv['ruv'])
    #print(mask)

    if method == 'fourier':
        # Apply the FFT-based beam mask and Fourier transform back to image:
        image_FFT_shift_mask = image_FFT_shift*mask
        image_SA = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift_mask))

    if method == 'convolve':
        # Apply convolution with Gaussian kernel:
        image_SA, kernel = convolve_to_beam(image, res2/60, res1/60, dxy, output = True)
        
    
    if plots:
        fig,ax = plt.subplots(2,2,figsize=(12,10))

        # Plot full-scale image:
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image,origin='lower',vmin=-0.5,vmax=0.5)

        # Plot simulated SA image:
        ax[1,1].set_title('Beam mask applied')
        ax[1,1].imshow(image_SA.real,origin='lower',vmin=-0.5,vmax=0.5)
          
        # Plot beam in Fourier or image plane:
        if method == 'fourier':
            ax[0,0].imshow(abs(image_FFT_shift)[idx['u1']:idx['u2']+1,idx['v1']:idx['v2']+1],
                           origin='lower',vmin=0,vmax=1e3,extent=extent)
            ax[0,1].imshow(abs(image_FFT_shift_mask)[idx['u1']:idx['u2']+1,idx['v1']:idx['v2']+1],
                           origin='lower',vmin=0,vmax=1e3,extent=extent)
        if method == 'convolve':
            ax[0,0].imshow(kernel,origin='lower')
            ax[0,1].imshow(kernel,origin='lower')

    
    return image_SA, mask

In [ ]:
def SA_deconvolve2(SA_image, SA_beam, noisecoeff, pix0, plots=True, *args,**kwargs):
    
    # FFT of input SA image:
    image_FFT = np.fft.fft2(SA_image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    # Get uv-plane axes:
    uv_freq = np.fft.fftshift(np.fft.fftfreq(SA_image.shape[0]))/dxy
    #uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    uv_m = uv_freq*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    # Make noise array for FFT of image:
    noiseRe = noisecoeff*np.random.normal(size=([SA_image.shape[0],SA_image.shape[1]]))
    noiseIm = noisecoeff*np.random.normal(size=([SA_image.shape[0],SA_image.shape[1]]))

    # Make noisy version of SA image FFT:
    #image_FFT_shift_noisy = np.empty_like(image_FFT_shift)
    image_FFT_shift_noisy  = image_FFT_shift + (noiseRe+1j*noiseIm) 

    # Deconvolve with SA beam:
    SA_beam[SA_beam<1e-100] = 1e-100
    image_FFT_deconvolved  = image_FFT_shift_noisy/SA_beam
   
    if plots:
        lims = 50
        print(image_FFT_shift.shape)
        fig,ax = plt.subplots(1,1,figsize=(8,4))
        ax.scatter(uv_m,abs(image_FFT_shift[pix0,:]))
        ax.scatter(uv_m,abs(image_FFT_shift_noisy[pix0,:]))
        ax.scatter(uv_m,abs(image_FFT_deconvolved[pix0,:]))
        ax.plot(uv_m,abs(image_FFT_shift[pix0,:]))
        ax.plot(uv_m,abs(image_FFT_shift_noisy[pix0,:]))
        ax.plot(uv_m,abs(image_FFT_deconvolved[pix0,:]))
        
        ax.set_ylim(1e-3,1e8)
        ax.set_yscale('log')
        ax.set_xlim(-lims,lims)
        ax.grid()

        # Plot simulated observation (SA image):
        fig,ax = plt.subplots(2,3,figsize=(12,10))
        ax[1,0].set_title('Original SA image')
        ax[1,0].imshow(SA_image.real,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=5e5,extent=extent)
        ax[0,0].set_xlim(-lims,lims)
        ax[0,0].set_ylim(-lims,lims)

        ax[1,1].set_title('Noisy SA image')
        ax[1,1].imshow(np.fft.ifft2(np.fft.ifftshift(image_FFT_shift_noisy)).real,origin='lower',vmin=0,vmax=30)
        ax[0,1].imshow(abs(image_FFT_shift_noisy),origin='lower',vmin=0,vmax=5e5,extent=extent)
        ax[0,1].set_xlim(-lims,lims)
        ax[0,1].set_ylim(-lims,lims)

        ax[1,2].set_title('deconvolved SA image')
        ax[1,2].imshow(np.fft.ifft2(np.fft.ifftshift(image_FFT_deconvolved)).real,origin='lower',vmin=0,vmax=30)
        ax[0,2].imshow(abs(image_FFT_deconvolved),origin='lower',vmin=0,vmax=5e5,extent=extent)
        ax[0,2].set_xlim(-lims,lims)
        ax[0,2].set_ylim(-lims,lims)

     
    return image_FFT_deconvolved

In [ ]:
def feather_ST(image,dxy,pix0,R1=12.9,R2=17.1,*arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv = get_uv_axis(image, dxy)

    mask = make_uv_mask('hp', R1, R2, uv['ruv'])
    
    image_FFT_shift = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_ST, mask, uv

In [ ]:
def feather_SA(image,FFT,dxy,pix0,R1=12.9,R2=17.1,*arg,**kwargs):

    uv = get_uv_axis(image, dxy)

    mask = make_uv_mask('lp', R1, R2, uv['ruv'])
    
    FFT = FFT*mask
    
    image_SA = np.fft.ifft2(np.fft.ifftshift(FFT))
       

    return image_SA, mask, uv

In [ ]:
def simple_gap(image,dxy,pix0,R1=14,R2=18,R3=9,R4=13,*arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    #uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)
    uv_m = 2*uv_freq*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask1 = 0.5*np.sin(b*(ruv - r0))+0.5
    mask1[ruv > R2] = 1
    mask1[ruv < R1] = 0
    
    b = np.pi/(R4-R3)
    r0 = (R3+R4)/2.
    mask2 = -0.5*np.sin(b*(ruv - r0))+0.5
    mask2[ruv > R4] = 0
    mask2[ruv < R3] = 1
    
    mask = mask1+mask2
    
    image_FFT_shift = image_FFT_shift*mask
    image_new = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_new, mask, uv_m

In [ ]:
def regular_image(data,j0,i0,nxy,taper=False,taper_width=50,*args,**kwargs):

    image = data[j0:j0+nxy,i0:i0+nxy]

    #dxy = np.round(hdr['CDELT2'],5)
    pix0 = int((nxy-1)/2)
    widxy = nxy*dxy
    
    if taper:      
        x, y = np.meshgrid(np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy), 
                       np.linspace(-(nxy-1)/2,(nxy-1)/2,nxy))
        r = np.sqrt(x**2+y**2)
        
        taper = gf(nxy/2-taper_width,taper_width,r)
        taper[np.where(r<=nxy/2-taper_width)] = 1
        image = image*taper

    print('Width of each pixel: '+str(dxy)+ ' deg.')
    print('Number of x and y pixels: '+str(nxy))
    print('Central pixel index: '+str(pix0))
    print('Width of image: '+str(widxy)+' deg.')
    
    return image, pix0, widxy

### Simulate ST and SA data

In [ ]:
image_full, pix0, widxy = regular_image(mapU_cgps, 500, 0, 1024)

#ST_map, ST_mask = ST_observe(image_full, dxy, pix0, R1=8.6, R2=12.9)
ST_map, ST_mask = ST_observe(image_full, dxy, pix0, R1=8.6+2.15, R2=12.9+2.15)
SA_map, SA_mask = SA_observe(image_full, dxy, pix0, R=9, method='fourier', bounds=50)

In [ ]:
print(ST_map.shape)
print(SA_map.shape)
print(8.6+2.15)
print(12.9+2.15)

### Deconvolve SA beam

In [ ]:
SA_noise = 100
SA_deconv_FFT = SA_deconvolve2(SA_map, SA_mask, SA_noise, pix0, plots=False)


### Feather

In [ ]:
ST_feather, ST_feather_mask, uv_ST = feather_ST(ST_map, dxy,pix0, R1=8.6, R2=17.1)
SA_feather, SA_feather_mask, uv_SA = feather_SA(image_full, SA_deconv_FFT, dxy, pix0, R1=8.6, R2=17.1)


In [ ]:
mapU_SA_sim = np.empty_like(mapU_cgps)
mapU_ST_sim = np.empty_like(mapU_cgps)
mapU_SA_feath = np.empty_like(mapU_cgps)
mapU_ST_feath = np.empty_like(mapU_cgps)

j0  = 500
i0  = 0
nxy = 1024

mapU_SA_sim[j0:j0+nxy,i0:i0+nxy] = SA_map.real
mapU_ST_sim[j0:j0+nxy,i0:i0+nxy] = ST_map.real
mapU_SA_feath[j0:j0+nxy,i0:i0+nxy] = SA_feather.real
mapU_ST_feath[j0:j0+nxy,i0:i0+nxy] = ST_feather.real

In [ ]:
mapU_SA_sim[mapU_SA_sim==0]=np.nan
mapU_ST_sim[np.isnan(mapU_SA_sim)]=np.nan
mapU_SA_feath[np.isnan(mapU_SA_sim)]=np.nan
mapU_ST_feath[np.isnan(mapU_SA_sim)]=np.nan
mapU_cgps[np.isnan(mapU_SA_sim)]=np.nan

fig,ax = plt.subplots(1,3,figsize=(12,3))
ax[0].hist(mapU_cgps.flatten(), bins=101, range=(-1,1));
ax[1].hist(mapU_SA_sim.flatten(), bins=101, range=(-1,1),alpha=0.5);
ax[1].hist(mapU_SA_feath.flatten(), bins=101, range=(-1,1),alpha=0.5);
ax[2].hist(mapU_ST_sim.flatten(), bins=101, range=(-1,1),alpha=0.5);
ax[2].hist(mapU_ST_feath.flatten(), bins=101, range=(-1,1),alpha=0.5);


In [ ]:
fig, ax = plt.subplots(1,5,figsize=(12,6))

ax[0].imshow(mapU_cgps, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')
ax[1].imshow(mapU_SA_sim, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')
ax[2].imshow(mapU_SA_feath, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')
ax[3].imshow(mapU_ST_sim, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')
ax[4].imshow(mapU_ST_feath, origin='lower', vmin=-0.6, vmax=0.6, cmap='rainbow')

In [ ]:
print(3.35--1.62)
print(163.33-158.35)

In [ ]:
def new_sim_plot_og():
    
    cx = SkyCoord([163.33,158.35], [0,0], frame=Galactic, unit="deg")
    cy = SkyCoord([160,160], [-1.62,3.35], frame=Galactic, unit="deg")

    # Create figure with adjusted spacing
    fig = plt.figure(figsize=(21,12.2))
    gs = GridSpec(2, 3, figure=fig)

    fs = 24
    
    axfull = fig.add_subplot(gs[0, 0], projection=wcs_cgps)
    axSA   = fig.add_subplot(gs[0, 1], projection=wcs_cgps)
    axST   = fig.add_subplot(gs[0, 2], projection=wcs_cgps)
    axcomb = fig.add_subplot(gs[1, 1], projection=wcs_cgps)
    axdiff = fig.add_subplot(gs[1, 2], projection=wcs_cgps)

    bbox = gs[1, 0].get_position(fig)
    smaller_box = [bbox.x0-bbox.width*0.33, 
                   bbox.y0-bbox.height*0.08, 
                   bbox.width*1.07, 
                   bbox.height*1.16]
    axcart = fig.add_axes(smaller_box)
    
    cmap = mpl.colormaps.get_cmap('rainbow') 
    cmap.set_bad(color='grey')

    vmax = 0.3

    im1 = axfull.imshow(mapU_cgps, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
    im2 = axSA.imshow(mapU_SA_sim, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
    im3 = axST.imshow(mapU_ST_sim, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
    im4 = axcomb.imshow(mapU_SA_feath + mapU_ST_feath, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
    im5 = axdiff.imshow(mapU_cgps - (mapU_SA_feath + mapU_ST_feath), origin='lower', cmap=cmap, vmin=-vmax/10, vmax=vmax/10)

    axcart.plot(uv_ST['uvm'],ST_feather_mask[pix0,:],color='red',linewidth=2)
    axcart.plot(uv_SA['uvm'],SA_feather_mask[pix0,:],color='blue',linewidth=2)
    axcart.plot(uv_ST['uvm'],ST_mask[pix0,:],color='k',linewidth=1)
    axcart.plot(uv_SA['uvm'],SA_mask[pix0,:],color='k',linewidth=1)
    axcart.plot(uv_SA['uvm'],SA_mask[pix0,:]+ST_mask[pix0,:],color='k',linewidth=3,linestyle='dashed')
    axcart.fill_between(uv_ST['uvm'],ST_mask[pix0,:],color='gray',alpha=0.2,linewidth=2)
    axcart.fill_between(uv_SA['uvm'],SA_mask[pix0,:],color='gray',alpha=0.2,linewidth=2)
    axcart.set_xlim(0,30)
    axcart.set_ylim(0,1.5)
    axcart.set_xlabel('Baseline (m)',fontsize=fs)
    axcart.tick_params(labelsize=fs)

    for ax in [axfull, axSA, axST, axcomb, axdiff]:
        ax.set_xlim(wcs_cgps.world_to_pixel(cx)[0])
        ax.set_ylim(wcs_cgps.world_to_pixel(cy)[1])
        ax.coords[1].set_ticks_visible(True)
        ax.coords[0].set_ticks_visible(True)
        if ax in [axfull, axcomb]:
            ax.coords[1].set_axislabel('Galactic Latitude', fontsize=fs, minpad=0.2)
            ax.coords[1].set_ticklabel_visible(True)
        else:
            ax.coords[1].set_axislabel(' ')
            ax.coords[1].set_ticklabel_visible(False)
        if ax in [axcomb, axdiff]:   
            ax.coords[0].set_axislabel('Galactic Longitude', fontsize=fs, minpad=0.6)
            ax.coords[0].set_ticklabel_visible(True)
        else:
            ax.coords[0].set_axislabel(' ')
            ax.coords[0].set_ticklabel_visible(False)
        ax.tick_params(axis='both', labelsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
            
    for spine in axcart.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)

    # Titles
    axfull.set_title('(a) Full-scale (\'true\') image',fontsize=fs, pad=10)
    axSA.set_title('(b) Single-antenna observation',fontsize=fs, pad=10)
    axST.set_title('(c) Aperture-synthesis observation',fontsize=fs, pad=10)
    axcart.set_title('(d) $uv$ coverage and filtering',fontsize=fs, pad=10)
    axcomb.set_title('(e) Combined reconstruction',fontsize=fs, pad=10)
    axdiff.set_title('(f) Residuals (true - combined)',fontsize=fs, pad=10)
    
    # To create colorbars with custom font size for labels
    def add_colorbar(im, ax):
        cbar = plt.colorbar(im, ax=ax, fraction=0.047, pad=0.01)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
        if ax in [axST, axdiff]:
            cbar.set_label(r'Stokes $U$ (K)', fontsize=fs)
        return cbar
        
    # Then use this function instead of direct fig.colorbar calls:
    add_colorbar(im1, axfull)
    add_colorbar(im2, axSA)
    add_colorbar(im3, axST)
    add_colorbar(im4, axcomb)
    add_colorbar(im5, axdiff)
    
    # Adjust edges to reduce whitespace
    plt.subplots_adjust(left=0.05, right=0.93, top=0.96, bottom=0.08, wspace=0.25, hspace=0.15)
    
    return fig


In [ ]:
new_sim_plot_og()
plt.savefig('../plots/uv_simulation_StokesU_with_coords_noforshorten.pdf')


In [ ]:
def hist_cals(hist, xarr):

    histx = (hist[1][:-1] + hist[1][1:]) / 2
    
    parameters, covariance = curve_fit(gauss, histx, hist[0]) 
    
    fwhm = 2*np.sqrt(2*np.log(2))*parameters[3]
    
    yfit = gauss(xarr,parameters[0],parameters[1],parameters[2],parameters[3])

    return fwhm, yfit

In [ ]:
def gauss(x, H, A, x0, sigma): 
    return H + A * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))

In [ ]:
print('cgps variability:')
print(np.nanstd(mapU_cgps))

print('')
print('DRAO ST + W06 variability:')
print(np.nanstd(mapU_w06_st))
print('Residuals variability:')
print(np.nanstd(mapU_cgps - mapU_w06_st))
print('Residuals variability compared to CGPS:')
print(100*np.nanstd(mapU_cgps - mapU_w06_st)/np.nanstd(mapU_cgps))

print('')
print('Simulated combination variability:')
print(np.nanstd(mapU_SA_feath + mapU_ST_feath))
print('Residuals variability:')
print(np.nanstd(mapU_cgps -(mapU_SA_feath + mapU_ST_feath)))
print('Residuals variability compared to CGPS:')
print(100*np.nanstd(mapU_cgps - (mapU_SA_feath + mapU_ST_feath))/np.nanstd(mapU_cgps))

In [ ]:
def new_sim_plot_claude():
    
    cx = SkyCoord([163.33,158.35], [0,0], frame=Galactic, unit="deg")
    cy = SkyCoord([160,160], [-1.62,3.35], frame=Galactic, unit="deg")

    # Create figure with adjusted spacing
    fig = plt.figure(figsize=(10,21))
    gs = GridSpec(5, 2, figure=fig)

    fs = 12
    
    axfull1 = fig.add_subplot(gs[0, 0], projection=wcs_cgps)
    #axfull2 = fig.add_subplot(gs[0, 1])
    axSA1   = fig.add_subplot(gs[1, 0], projection=wcs_cgps)
    axSA2   = fig.add_subplot(gs[1, 1], projection=wcs_cgps)
    axST1   = fig.add_subplot(gs[2, 0], projection=wcs_cgps)
    axST2   = fig.add_subplot(gs[2, 1], projection=wcs_st)
    axcomb1 = fig.add_subplot(gs[3, 0], projection=wcs_cgps)
    axcomb2 = fig.add_subplot(gs[3, 1], projection=wcs_w06_st)
    axdiff1 = fig.add_subplot(gs[4, 0], projection=wcs_cgps)
    axdiff2 = fig.add_subplot(gs[4, 1], projection=wcs_cgps)

    bbox = gs[0, 1].get_position(fig)
    smaller_box = [bbox.x0+bbox.width*0.08, 
                   bbox.y0+bbox.height*0.7, 
                   bbox.width*1.05, 
                   bbox.height*1.05]
    axfull2 = fig.add_axes(smaller_box)
    
    cmap = mpl.colormaps.get_cmap('rainbow') 
    cmap.set_bad(color='grey')

    vmax = 0.3

    # Store the image objects to add colorbars later
    im1 = axfull1.imshow(mapU_cgps, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)

    axfull2.plot(uv_ST['uvm'],ST_feather_mask[pix0,:],color='red',linewidth=2)
    axfull2.plot(uv_SA['uvm'],SA_feather_mask[pix0,:],color='blue',linewidth=2)
    axfull2.plot(uv_ST['uvm'],ST_mask[pix0,:],color='k',linewidth=1)
    axfull2.plot(uv_SA['uvm'],SA_mask[pix0,:],color='k',linewidth=1)
    axfull2.plot(uv_SA['uvm'],SA_mask[pix0,:]+ST_mask[pix0,:],color='k',linewidth=3,linestyle='dashed')
    axfull2.fill_between(uv_ST['uvm'],ST_mask[pix0,:],color='gray',alpha=0.2,linewidth=2)
    axfull2.fill_between(uv_SA['uvm'],SA_mask[pix0,:],color='gray',alpha=0.2,linewidth=2)
    axfull2.set_xlim(0,30)
    axfull2.set_ylim(0,1.5)
    axfull2.set_xlabel('Baseline (m)',fontsize=fs)
    axfull2.tick_params(labelsize=fs)

    im2 = axSA1.imshow(mapU_SA_sim, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
    im3 = axSA2.imshow(mapU_w06_rgrd, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)

    im4 = axST1.imshow(mapU_ST_sim, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
    im5 = axST2.imshow(mapU_st, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)

    im6 = axcomb1.imshow(mapU_SA_sim + mapU_ST_sim, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
    im7 = axcomb2.imshow(mapU_w06_st, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)

    im8 = axdiff1.imshow(mapU_cgps - (mapU_SA_sim + mapU_ST_sim), origin='lower', cmap=cmap, vmin=-vmax/10, vmax=vmax/10)
    im9 = axdiff2.imshow(mapU_cgps - mapU_w06_st, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)

    for ax in [axfull1, axSA1, axST1, axcomb1, axcomb2, axdiff1, axdiff2]:
        ax.set_xlim(wcs_cgps.world_to_pixel(cx)[0])
        ax.set_ylim(wcs_cgps.world_to_pixel(cy)[1])
    
    axSA2.set_xlim(wcs_cgps.world_to_pixel(cx)[0])
    axSA2.set_ylim(wcs_cgps.world_to_pixel(cy)[1])

    axST2.set_xlim(wcs_st.world_to_pixel(cx)[0])
    axST2.set_ylim(wcs_st.world_to_pixel(cy)[1])

    # WCS-compatible method to hide tick labels without removing tick marks
    for ax in [axfull1, axSA1, axST1, axcomb1, axSA2, axST2, axcomb2]:
        ax.coords[0].set_ticklabel_visible(False)
        ax.coords[0].set_ticks_visible(True)
    
    # For all axes, set the y-axis labels
    for ax in [axfull1, axSA1, axST1, axcomb1, axdiff1]:
        ax.coords[1].set_axislabel('Galactic Latitude', fontsize=fs)
        
    # Hide y-axis labels for right column
    for ax in [axfull2, axSA2, axST2, axcomb2, axdiff2]:
        if hasattr(ax, 'coords'):  # Only WCS axes have coords attribute
            ax.coords[1].set_axislabel(' ')
        else:
            ax.set_ylabel(' ')
    
    # Set x-axis labels for bottom row only
    axdiff1.coords[0].set_axislabel('Galactic Longitude', fontsize=fs)
    axdiff2.coords[0].set_axislabel('Galactic Longitude', fontsize=fs)

    for ax in [axfull1, axSA1, axST1, axcomb1, axdiff1, axSA2, axST2, axcomb2, axdiff2]:
        #ax.set_ticklabel_size(fs)  # x-axis tick labels
        #ax.set_ticklabel_size(fs)  # y-axis tick labels
        #else:
        #    # For regular matplotlib axes (lik
        ax.tick_params(axis='both', labelsize=fs)

    # Titles
    axfull1.set_title('(a) Full-scale image (CGPS)',fontsize=fs)
    axfull2.set_title('(b) $uv$ coverage and filtering',fontsize=fs)
    axSA1.set_title('(c) Single-antenna simulated',fontsize=fs)
    axSA2.set_title('(d) True single-antenna (W06)',fontsize=fs)
    axST1.set_title('(e) Aperture-synthesis simulated',fontsize=fs)
    axST2.set_title('(f) True aperture-synthesis (DRAO ST)',fontsize=fs)
    axcomb1.set_title('(g) Combined simulated',fontsize=fs)
    axcomb2.set_title('(h) True combined',fontsize=fs)
    axdiff1.set_title('(i) Simulated observation residuals',fontsize=fs)
    axdiff2.set_title('(j) True residuals',fontsize=fs)
    
    # To create colorbars with custom font size for labels
    def add_colorbar(im, ax):
        cbar = plt.colorbar(im, ax=ax, fraction=0.055, pad=0.01)
        cbar.ax.tick_params(labelsize=fs)
        return cbar
        
    # Then use this function instead of direct fig.colorbar calls:
    add_colorbar(im1, axfull1)
    # No colorbar for axfull2
    add_colorbar(im2, axSA1)
    add_colorbar(im3, axSA2)
    add_colorbar(im4, axST1)
    add_colorbar(im5, axST2)
    add_colorbar(im6, axcomb1)
    add_colorbar(im7, axcomb2)
    add_colorbar(im8, axdiff1)
    add_colorbar(im9, axdiff2)
    
    # Adjust edges to reduce whitespace
    plt.subplots_adjust(left=0.04, right=0.95, top=0.98, bottom=0.04, wspace=0.15, hspace=0.15)
    
    return fig



In [ ]:
new_sim_plot_claude()
#plt.savefig('../plots/uv_simulation_StokesU_referee_suggestion.pdf')
